# Notebook 1 — Data Cleaning

**Business objective:** Before any analysis, I need clean and reliable data.
This notebook loads the three raw Inside Airbnb files, fixes the columns I
will use, handles missing values, and saves clean files for the rest of the
project.

I keep only the columns the project actually needs. The raw listings file has
79 columns, but most are not relevant to pricing, occupancy, or guest
satisfaction.

**Steps in this notebook:**
1. Load the raw data
2. Clean the listings data
3. Clean the calendar data
4. Clean the reviews data
5. Save the clean files

## Step 1 — Load the raw data

The three files come straight from Inside Airbnb as gzipped CSV files.
A gzipped CSV is just a normal CSV that has been compressed to save space,
and pandas can read it directly without me unzipping it first. I load each
file and check its shape (the number of rows and columns) so I know how much
data I am working with.

In [1]:
import pandas as pd

# The raw files live in the data/raw folder, one level up from this notebook.
raw_path = "../data/raw/"

listings = pd.read_csv(raw_path + "listings.csv.gz")
calendar = pd.read_csv(raw_path + "calendar.csv.gz")
reviews = pd.read_csv(raw_path + "reviews.csv.gz")

print("Listings:", listings.shape)
print("Calendar:", calendar.shape)
print("Reviews:", reviews.shape)

Listings: (14274, 79)
Calendar: (5210011, 7)
Reviews: (635471, 6)


The shapes confirm the structure I expected: listings has one row per
rental, calendar has one row per listing per day, and reviews has one row per
guest review.

## Step 2 — Clean the listings data

The listings file is the most important one. I start by keeping only the
columns I need for the analysis, which makes the data easier to work with and
easier to explain.

A note on two column names that use abbreviations:
- `number_of_reviews_ltm` — "ltm" means "last twelve months", so this is the
  number of reviews a listing received in the past year.
- `estimated_occupancy_l365d` — "l365d" means "last 365 days", so this is
  Inside Airbnb's estimate of how many nights the listing was booked in the
  past year.

In [2]:
columns_to_keep = [
    "id", "name", "host_id", "host_is_superhost",
    "neighbourhood_cleansed", "neighbourhood_group_cleansed",
    "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bedrooms", "beds",
    "price", "minimum_nights", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm",
    "estimated_occupancy_l365d", "estimated_revenue_l365d",
    "review_scores_rating", "review_scores_cleanliness",
    "review_scores_location", "review_scores_value",
    "reviews_per_month", "instant_bookable",
]

listings = listings[columns_to_keep]
print("Listings now has", listings.shape[1], "columns")

Listings now has 26 columns


### Clean the price column

The price column comes in as text, for example `$105.00`. I cannot do
calculations on text, so I need to turn it into a number. I do this in three
small steps: force the column to text (this is safe even when some values are
blank), remove the dollar sign and the thousands comma, then convert the
result to a number.

The `errors="coerce"` setting means that if any value still cannot be turned
into a number, pandas replaces it with a missing value (NaN) instead of
stopping with an error. This is exactly what I want, because a few odd values
should not break the whole step.

In [3]:
listings["price"] = listings["price"].astype(str)
listings["price"] = listings["price"].str.replace("$", "", regex=False)
listings["price"] = listings["price"].str.replace(",", "", regex=False)
listings["price"] = pd.to_numeric(listings["price"], errors="coerce")

print("Price column type is now:", listings["price"].dtype)
print(listings["price"].describe().round(1))

Price column type is now: float64
count     9264.0
mean       201.2
std       1657.0
min          5.0
25%         70.0
50%        104.0
75%        160.0
max      50000.0
Name: price, dtype: float64


### Check missing values

I count how many values are missing in each column. This tells me which
columns I can trust as they are, and which ones need a decision from me.

In [4]:
missing = listings.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing values by column:")
print(missing)

Missing values by column:
estimated_revenue_l365d      5010
price                        5010
beds                         4995
review_scores_value          3318
review_scores_location       3317
review_scores_rating         3314
review_scores_cleanliness    3314
reviews_per_month            3314
bedrooms                     2024
host_is_superhost             164
dtype: int64


There are two main gaps, and I handle them differently on purpose:

- **Price** is missing for many listings. These are listings that simply do
  not have a nightly price set in this snapshot. I will not invent a price for
  them. Instead, in the next cell, I create a flag so I can include or exclude
  them whenever I choose.
- **Review scores** are missing for listings that have no reviews yet. This is
  normal — a brand new listing has no rating. I leave these as missing rather
  than filling them with a made-up score, which would distort the averages.

In [5]:
# A True/False flag marking which listings have a price.
# This lets me separate priced from unpriced listings later without
# deleting any rows.
listings["has_price"] = listings["price"].notna()

print("Listings with a price:", listings["has_price"].sum())
print("Listings without a price:", (~listings["has_price"]).sum())

Listings with a price: 9264
Listings without a price: 5010


### Tidy the superhost column

The `host_is_superhost` column stores the letter "t" for true and "f" for
false. I convert these to the clearer labels "Yes" and "No" so the values are
easy to read in charts and in the Tableau dashboard. A small number of
listings have no value here, which I label "Unknown" rather than dropping, so
the total listing count stays correct.

In [6]:
listings["host_is_superhost"] = listings["host_is_superhost"].map(
    {"t": "Yes", "f": "No"}
)
listings["host_is_superhost"] = listings["host_is_superhost"].fillna("Unknown")

print(listings["host_is_superhost"].value_counts())

host_is_superhost
No         10578
Yes         3532
Unknown      164
Name: count, dtype: int64


The `instant_bookable` column uses the same "t" and "f" values, so I
convert it the same way.

In [7]:
listings["instant_bookable"] = listings["instant_bookable"].map(
    {"t": "Yes", "f": "No"}
)
print(listings["instant_bookable"].value_counts())

instant_bookable
No     9908
Yes    4366
Name: count, dtype: int64


### Create an occupancy rate column

The `estimated_occupancy_l365d` column gives the estimated number of nights a
listing was booked in the last 365 days. A raw night count is hard to read at
a glance, so I turn it into a percentage of the year: booked nights divided by
365, multiplied by 100. For example, 91 booked nights becomes about 25
percent, meaning the listing was booked roughly a quarter of the year.

In [8]:
listings["occupancy_rate"] = (listings["estimated_occupancy_l365d"] / 365) * 100
listings["occupancy_rate"] = listings["occupancy_rate"].round(1)

print(listings["occupancy_rate"].describe().round(1))

count    14274.0
mean        21.2
std         27.8
min          0.0
25%          0.0
50%          3.6
75%         44.9
max         69.9
Name: occupancy_rate, dtype: float64


### Check for duplicate listings

Each listing should appear only once. I confirm this by counting how many
listing IDs are duplicated. A result of zero means every listing is unique.

In [9]:
duplicate_count = listings["id"].duplicated().sum()
print("Duplicate listing ids:", duplicate_count)

Duplicate listing ids: 0


## Step 3 — Clean the calendar data

The calendar file records whether each listing is available on each date.
Two columns need cleaning: the `available` column uses "t" and "f", and the
`price` column is text just like it was in the listings file. I also convert
the `date` column from text into a real date so it can be used for time-based
work later.

In [10]:
# Convert the "t"/"f" availability values into True/False.
calendar["is_available"] = calendar["available"].map({"t": True, "f": False})

# Clean the price column the same way as before. Most calendar rows have no
# price, so I force the column to text first to keep the steps safe.
calendar["price"] = calendar["price"].astype(str)
calendar["price"] = calendar["price"].str.replace("$", "", regex=False)
calendar["price"] = calendar["price"].str.replace(",", "", regex=False)
calendar["price"] = pd.to_numeric(calendar["price"], errors="coerce")

# Convert the date text into a real date value.
calendar["date"] = pd.to_datetime(calendar["date"])

print(calendar[["listing_id", "date", "is_available", "price"]].head())

   listing_id       date  is_available  price
0     1358910 2025-09-24         False    NaN
1     1358910 2025-09-25         False    NaN
2     1358910 2025-09-26          True    NaN
3     1358910 2025-09-27          True    NaN
4     1358910 2025-09-28          True    NaN


## Step 4 — Clean the reviews data

The reviews file is used mainly to count how many reviews each listing
receives over time. I convert the `date` column into a real date, remove any
reviews that have no date (since they cannot be placed in time), and add a
`review_year` column so I can look at recent activity later.

In [11]:
reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")

# Remove reviews with no date, because they cannot be used in any analysis
# that depends on when the review was written.
rows_before = len(reviews)
reviews = reviews.dropna(subset=["date"])
print("Dropped", rows_before - len(reviews), "reviews with no date")

# Pull the year out of each review date into its own column.
reviews["review_year"] = reviews["date"].dt.year
print(reviews["review_year"].value_counts().sort_index().tail(6))

Dropped 0 reviews with no date
review_year
2020     28608
2021     39305
2022     77312
2023     95640
2024    121499
2025    101825
Name: count, dtype: int64


## Step 5 — Save the clean files

I save the three cleaned files into the `data/clean` folder. The rest of
the project loads these cleaned files instead of the raw ones, so the cleaning
work only needs to happen once.

In [12]:
clean_path = "../data/clean/"

listings.to_csv(clean_path + "listings_clean.csv", index=False)
calendar.to_csv(clean_path + "calendar_clean.csv", index=False)
reviews.to_csv(clean_path + "reviews_clean.csv", index=False)

print("Saved clean files:")
print("- listings_clean.csv :", listings.shape)
print("- calendar_clean.csv :", calendar.shape)
print("- reviews_clean.csv  :", reviews.shape)

Saved clean files:
- listings_clean.csv : (14274, 28)
- calendar_clean.csv : (5210011, 8)
- reviews_clean.csv  : (635471, 7)


## Insight summary

The data is now clean and ready for analysis:

- **Listings:** 14,274 rows. Price is now a proper number. About 9,300
  listings have a nightly price; the rest do not have one set in this
  snapshot, and I flagged them rather than guessing values.
- **Calendar:** availability and price are cleaned, and dates are real dates.
- **Reviews:** dates are cleaned and a review year column was added.

## Business recommendation

For any pricing analysis, I will use only listings that have a real price, and
I will remove extreme values in the next notebook so the averages stay
realistic. Occupancy and review scores can use all listings, because a missing
review score simply means a listing has no reviews yet.